# Lab 13 - Part 2: Search Functions and Retrieval Approaches

**Name:** Abbas Abdelaziz Al-Tunayan  
**Student ID:** 2240004112  
**Course:** Natural Language Processing

**Learning goals:** Reuse the retrieval pipeline in a function, explore different queries, and compare sparse and dense retrieval.

Run the cells from top to bottom. The setup recreates all required data, so Part 1 does not need to be open or executed.

## Setup — Run this cell first

The cell below recreates the small document collection and TF-IDF vectors used in Part 1.

In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

documents = [
    "Transformers use self-attention to understand relationships between words.",
    "TF-IDF represents documents using word importance scores.",
    "Word2Vec learns dense vector representations of words.",
    "Named entity recognition identifies names, places, and organizations in text.",
    "Information retrieval finds relevant documents for a user query.",
    "Cosine similarity measures how similar two text vectors are."
]

vectorizer = TfidfVectorizer(stop_words="english")
document_vectors = vectorizer.fit_transform(documents)

## 1. Make a simple search function

Now combine the steps so you can search the document collection using any query.

In [2]:
def search(query, k=2):
    query_vector = vectorizer.transform([query])
    scores = cosine_similarity(query_vector, document_vectors)[0]
    top_ids = scores.argsort()[::-1][:k]

    for i in top_ids:
        print(f"Score: {scores[i]:.3f} | {documents[i]}")

search("How can we measure similarity between text vectors?")

Score: 0.686 | Cosine similarity measures how similar two text vectors are.
Score: 0.148 | Named entity recognition identifies names, places, and organizations in text.


## 2. Sparse vs. dense retrieval (conceptual comparison)

| Sparse retrieval | Dense retrieval |
|---|---|
| Represents text using word-based features, e.g., TF-IDF | Represents text using learned embeddings |
| Works well when important query words appear in documents | Can identify similar meanings even when different words are used |
| Simple and lightweight | Usually requires a pretrained embedding model |

**Important limitation:** TF-IDF mainly matches words, not meanings. A query about *"automobiles"* may not retrieve a document about *"cars"* unless the words also overlap in other ways.

In modern RAG, retrieval can be sparse, dense, or a combination of both.

## 3. Does a high-ranked result always mean a good match?

The search function returns the **highest-scoring** documents, even when all the scores are zero. A high *rank* does not always mean a high *similarity score*.

For example, our collection is about NLP. What happens when we search for an unrelated topic like `"volcano eruption"`?

In [3]:
search("volcano eruption", k=2)

# Compare it with a relevant query
print("--- Relevant query ---")
search("cosine similarity", k=2)

Score: 0.000 | Cosine similarity measures how similar two text vectors are.
Score: 0.000 | Information retrieval finds relevant documents for a user query.
--- Relevant query ---
Score: 0.594 | Cosine similarity measures how similar two text vectors are.
Score: 0.000 | Information retrieval finds relevant documents for a user query.


**Observation:** `search("volcano eruption", k=2)` still prints two results,
but both scores are zero. Neither query term occurs in the vocabulary. Their
rank is only the result of sorting tied scores, so it does not show relevance.
For `"cosine similarity"`, the cosine-similarity note has a positive score and
ranks first. A high position alone is insufficient; the score must also be checked.

## 4. Improve search: skip results with a score of zero

We can improve the function by printing only documents with a **positive similarity score**.

In [4]:
def search_filtered(query, k=2):
    query_vector = vectorizer.transform([query])
    scores = cosine_similarity(query_vector, document_vectors)[0]
    top_ids = scores.argsort()[::-1][:k]

    found = False
    for i in top_ids:
        if scores[i] > 0:
            print(f"Score: {scores[i]:.3f} | {documents[i]}")
            found = True
    if not found:
        print("No matching documents found.")

search_filtered("volcano eruption")
print("---")
search_filtered("cosine similarity")

No matching documents found.
---
Score: 0.594 | Cosine similarity measures how similar two text vectors are.


## 5. Practice questions

1. Use `search()` to find a note about **named entity recognition**. Which document ranks first?
2. Add `"BERT is a transformer model that uses bidirectional context."` to `documents`, rebuild the TF-IDF vectors using `vectorizer.fit_transform(documents)`, and search for `"transformer models"`. Why was rebuilding necessary?

In [5]:
search("named entity recognition", k=2)

Score: 0.625 | Named entity recognition identifies names, places, and organizations in text.
Score: 0.000 | Cosine similarity measures how similar two text vectors are.


### Practice question 1 - Answer

Document **3** (the fourth document) ranks first:

> Named entity recognition identifies names, places, and organizations in text.

Its cosine similarity is approximately **0.625**, because it contains the query
terms **named**, **entity**, and **recognition**. The other displayed result has
a score of zero and is not a matching note.

### Practice question 2 - Add BERT and rebuild the index

Add the exact sentence specified in the question, rebuild the TF-IDF matrix,
and search for `"transformer models"`.

In [6]:
new_document = "BERT is a transformer model that uses bidirectional context."

# Avoid duplicate documents if this cell is run again.
if new_document not in documents:
    documents.append(new_document)

print("Stored documents:", len(documents))
print("Matrix shape before rebuilding:", document_vectors.shape)

document_vectors = vectorizer.fit_transform(documents)

print("Matrix shape after rebuilding:", document_vectors.shape)
print("The vocabulary contains 'transformer':",
      "transformer" in vectorizer.vocabulary_)
print('\nSearch results for "transformer models":')
search("transformer models", k=2)

print('\nResults with positive similarity only:')
search_filtered("transformer models", k=2)

Stored documents: 7
Matrix shape before rebuilding: (6, 39)
Matrix shape after rebuilding: (7, 45)
The vocabulary contains 'transformer': True

Search results for "transformer models":
Score: 0.408 | BERT is a transformer model that uses bidirectional context.
Score: 0.000 | Cosine similarity measures how similar two text vectors are.

Results with positive similarity only:
Score: 0.408 | BERT is a transformer model that uses bidirectional context.


**Answer:** The new BERT note, Document **6** (the seventh document), ranks first
with a cosine similarity of approximately **0.408**.

Rebuilding was necessary for three reasons:

1. Appending to `documents` only changes the Python list. The old TF-IDF matrix
   still has six rows and cannot retrieve the new note.
2. Fitting again learns vocabulary terms from the new document, including
   **"transformer"**, **"BERT"**, and **"bidirectional"**.
3. IDF weights depend on the collection size and document frequencies, so the
   weights for the whole collection must be recomputed.

The matrix changes from **6 x 39** to **7 x 45**. Queries must then be transformed
with this same refitted vectorizer so query and document features stay aligned.

The query matches the singular word **"transformer"** in the BERT note.
**"models"** is absent from the vocabulary and is ignored. The original first
document contains **"Transformers"**, which is a different token in this basic
TF-IDF pipeline. A second zero-score result from `search()` is not a meaningful
match; `search_filtered()` removes it.

## Summary

The search function ranks the document collection using TF-IDF and cosine
similarity. Sparse retrieval relies on shared word features; dense retrieval
uses learned embeddings and can match semantically related wording. A result
can rank highly even when its score is zero, so filtering avoids displaying
documents with no lexical overlap. Updating the collection requires rebuilding
the fitted vocabulary and document vectors before searching it again.